# Amazon Search / LTR — Week 4 V2 One-Click Colab
## CLIP Image Embeddings + Multimodal Retrieval

This notebook extends **Week 4 V1** with product-image retrieval and fixed-weight multimodal fusion.

### Inputs
Upload:
1. `amazon_week1_v4_processed_data.zip`
2. `amazon_week4_v1_sbert_faiss_results.zip`

### Pipeline
- Load Week 1 10k catalog and image URLs
- Reuse V1 SBERT product embeddings and fixed test queries
- Download available product images
- Encode product images with **CLIP**
- Build a CLIP **FAISS** image index
- Encode search queries with CLIP text encoder
- Evaluate **CLIP-only**
- Fuse **SBERT text similarity + CLIP image similarity**
- Compare:
  - BM25
  - SBERT + FAISS
  - CLIP image retrieval
  - SBERT + CLIP multimodal retrieval
- Measure image coverage and retrieval latency
- Export all results

### Important evaluation choice
The default fusion weight is fixed **before evaluation**:

`multimodal_score = 0.70 * SBERT + 0.30 * CLIP`

For products without a usable image, the image component falls back conservatively so text retrieval still works.

In [ ]:
# ============================================================
# 0. CONFIG
# ============================================================

TOP_K = 50
CLIP_MODEL = "openai/clip-vit-base-patch32"

# Fixed, pre-declared fusion weight:
TEXT_WEIGHT = 0.70
IMAGE_WEIGHT = 0.30

# Image pipeline
IMAGE_SIZE_TIMEOUT = 8
DOWNLOAD_WORKERS = 32
CLIP_BATCH_SIZE = 64

# None = attempt all non-empty image URLs in the 10k catalog.
# For a quick test, set e.g. 1000; for final run keep None.
MAX_IMAGE_DOWNLOADS = None

LATENCY_SAMPLE_SIZE = 30
RANDOM_SEED = 42

print({
    "TOP_K": TOP_K,
    "CLIP_MODEL": CLIP_MODEL,
    "TEXT_WEIGHT": TEXT_WEIGHT,
    "IMAGE_WEIGHT": IMAGE_WEIGHT,
    "MAX_IMAGE_DOWNLOADS": MAX_IMAGE_DOWNLOADS,
})

In [ ]:
# ============================================================
# 1. INSTALL DEPENDENCIES
# ============================================================

import sys, subprocess

packages = [
    "transformers>=4.45",
    "faiss-cpu>=1.8",
    "pyarrow>=15",
    "pillow>=10",
    "requests>=2.31",
    "tqdm>=4.66",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("Dependencies installed.")

In [ ]:
# ============================================================
# 2. IMPORTS + DIRECTORIES
# ============================================================

import os, io, re, gc, sys, json, time, math, shutil, random, zipfile
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from PIL import Image, ImageFile
from tqdm.auto import tqdm

import torch
import faiss
from transformers import CLIPModel, CLIPProcessor
from sklearn.metrics import ndcg_score

ImageFile.LOAD_TRUNCATED_IMAGES = True
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
INPUT_DIR = ROOT / "week4_v2_inputs"
EXTRACT_DIR = ROOT / "week4_v2_extracted"
IMAGE_DIR = ROOT / "week4_v2_images"
OUTPUT_DIR = ROOT / "week4_v2_outputs"

for p in [INPUT_DIR, EXTRACT_DIR, IMAGE_DIR, OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# 3. UPLOAD WEEK 1 ZIP + WEEK 4 V1 RESULT ZIP
# ============================================================

is_colab = False
try:
    from google.colab import files
    is_colab = True
except Exception:
    files = None

if is_colab:
    print("Upload BOTH files:")
    print("1) amazon_week1_v4_processed_data.zip")
    print("2) amazon_week4_v1_sbert_faiss_results.zip")
    uploaded = files.upload()
    for filename, content in uploaded.items():
        with open(INPUT_DIR / filename, "wb") as f:
            f.write(content)
else:
    print("Place both ZIP files in:", INPUT_DIR)

zip_files = sorted(INPUT_DIR.glob("*.zip"))

if len(zip_files) < 2:
    for z in ROOT.glob("*.zip"):
        if z.parent != INPUT_DIR:
            dst = INPUT_DIR / z.name
            if not dst.exists():
                shutil.copy2(z, dst)
    zip_files = sorted(INPUT_DIR.glob("*.zip"))

print("\nZIPs found:")
for z in zip_files:
    print(" -", z.name)

if len(zip_files) < 2:
    raise FileNotFoundError("Need Week 1 processed-data ZIP and Week 4 V1 results ZIP.")

In [ ]:
# ============================================================
# 4. EXTRACT + AUTO-DETECT FILES
# ============================================================

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

for zpath in zip_files:
    out = EXTRACT_DIR / zpath.stem
    out.mkdir(parents=True, exist_ok=True)
    try:
        with zipfile.ZipFile(zpath, "r") as zf:
            zf.extractall(out)
        print("Extracted:", zpath.name)
    except zipfile.BadZipFile:
        pass

def find_one(filename):
    matches = list(EXTRACT_DIR.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"Missing required file: {filename}")
    return matches[0]

PRODUCTS_PATH = find_one("cleaned_products_10k.parquet")
QPR_PATH = find_one("query_product_relevance.parquet")

SBERT_EMB_PATH = find_one("week4_sbert_product_embeddings.npy")
TEST_QUERIES_PATH = find_one("week4_test_queries_used.csv")
V1_METRICS_PATH = find_one("week4_text_retrieval_metrics.csv")
V1_BM25_TOP50_PATH = find_one("week4_bm25_top50.csv")
V1_SBERT_TOP50_PATH = find_one("week4_sbert_top50.csv")
V1_LATENCY_PATH = find_one("week4_latency_results.csv")

print("All required files detected.")

In [ ]:
# ============================================================
# 5. LOAD DATA + ALIGN V1 EMBEDDINGS
# ============================================================

products = pd.read_parquet(PRODUCTS_PATH)
qpr = pd.read_parquet(QPR_PATH)
test_queries = pd.read_csv(TEST_QUERIES_PATH)

products["product_id"] = products["product_id"].astype(str)
qpr["product_id"] = qpr["product_id"].astype(str)
qpr["query_id"] = qpr["query_id"].astype(str)
test_queries["query_id"] = test_queries["query_id"].astype(str)

products = products.drop_duplicates("product_id").reset_index(drop=True)

sbert_embeddings = np.load(SBERT_EMB_PATH).astype("float32")
if len(products) != len(sbert_embeddings):
    raise ValueError(
        f"Catalog has {len(products)} products but V1 SBERT embeddings have "
        f"{len(sbert_embeddings)} rows. Product order must match V1."
    )

product_ids = products["product_id"].tolist()
idx_to_pid = np.array(product_ids, dtype=object)
pid_to_idx = {pid:i for i,pid in enumerate(product_ids)}
title_map = products.set_index("product_id")["title"].fillna("").astype(str).to_dict()

if "image_url" not in products.columns:
    raise ValueError("Week 1 catalog has no image_url column.")

products["image_url"] = products["image_url"].fillna("").astype(str).str.strip()

gold = qpr[qpr["query_id"].isin(test_queries["query_id"])].copy()

print("Products:", len(products))
print("Test queries:", len(test_queries))
print("Non-empty image URLs:", int((products["image_url"].str.len() > 0).sum()))
print("Image URL coverage:", round((products["image_url"].str.len() > 0).mean()*100, 2), "%")

In [ ]:
# ============================================================
# 6. DOWNLOAD PRODUCT IMAGES
# ============================================================

# Recreate clean image directory
if IMAGE_DIR.exists():
    shutil.rmtree(IMAGE_DIR)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

image_candidates = products.loc[
    products["image_url"].str.startswith(("http://", "https://")),
    ["product_id", "image_url"]
].copy()

if MAX_IMAGE_DOWNLOADS is not None:
    image_candidates = image_candidates.head(MAX_IMAGE_DOWNLOADS)

session_headers = {
    "User-Agent": "Mozilla/5.0 (compatible; AmazonSearchResearch/1.0)"
}

def download_one(row):
    pid = str(row.product_id)
    url = str(row.image_url)
    out = IMAGE_DIR / f"{pid}.jpg"

    try:
        r = requests.get(
            url,
            timeout=IMAGE_SIZE_TIMEOUT,
            headers=session_headers,
            stream=True
        )
        if r.status_code != 200:
            return pid, False, f"http_{r.status_code}"

        content = r.content
        img = Image.open(io.BytesIO(content)).convert("RGB")
        # Validate and save standardized JPEG.
        img.save(out, format="JPEG", quality=90)
        return pid, True, ""
    except Exception as e:
        return pid, False, type(e).__name__

records = []

print(f"Attempting {len(image_candidates)} image downloads...")

with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as ex:
    futures = [ex.submit(download_one, r) for r in image_candidates.itertuples(index=False)]
    for f in tqdm(as_completed(futures), total=len(futures)):
        pid, ok, err = f.result()
        records.append({"product_id": pid, "download_ok": ok, "error": err})

download_status = pd.DataFrame(records)
download_status.to_csv(OUTPUT_DIR / "week4_clip_download_status.csv", index=False)

ok_ids = set(download_status.loc[download_status["download_ok"], "product_id"].astype(str))
products["image_available"] = products["product_id"].isin(ok_ids)

print("Successful images:", int(products["image_available"].sum()))
print("Successful image coverage:", round(products["image_available"].mean()*100, 2), "%")

In [ ]:
# ============================================================
# 7. LOAD CLIP + ENCODE PRODUCT IMAGES
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading:", CLIP_MODEL)
clip_model = CLIPModel.from_pretrained(CLIP_MODEL).to(device)
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL)
clip_model.eval()

available_rows = products[products["image_available"]].copy()
available_indices = available_rows.index.to_numpy()
available_pids = available_rows["product_id"].tolist()

all_image_embeddings = []

def load_image(pid):
    return Image.open(IMAGE_DIR / f"{pid}.jpg").convert("RGB")

t0 = time.perf_counter()

with torch.no_grad():
    for start in tqdm(range(0, len(available_pids), CLIP_BATCH_SIZE)):
        batch_pids = available_pids[start:start+CLIP_BATCH_SIZE]
        images = [load_image(pid) for pid in batch_pids]

        inputs = clip_processor(images=images, return_tensors="pt")
        pixel_values = inputs["pixel_values"].to(device)

        feats = clip_model.get_image_features(pixel_values=pixel_values)
        feats = feats / feats.norm(dim=-1, keepdim=True)
        all_image_embeddings.append(feats.cpu().numpy().astype("float32"))

clip_image_embeddings = np.vstack(all_image_embeddings)
clip_encode_seconds = time.perf_counter() - t0

np.save(OUTPUT_DIR / "week4_clip_image_embeddings.npy", clip_image_embeddings)
np.save(OUTPUT_DIR / "week4_clip_catalog_indices.npy", available_indices)

pd.DataFrame({
    "clip_row": np.arange(len(available_pids)),
    "catalog_index": available_indices,
    "product_id": available_pids
}).to_csv(OUTPUT_DIR / "week4_clip_embedding_map.csv", index=False)

print("CLIP image embedding shape:", clip_image_embeddings.shape)
print("Encoding time (sec):", round(clip_encode_seconds, 2))

In [ ]:
# ============================================================
# 8. BUILD CLIP FAISS IMAGE INDEX
# ============================================================

clip_dim = clip_image_embeddings.shape[1]
clip_index = faiss.IndexFlatIP(clip_dim)

t0 = time.perf_counter()
clip_index.add(clip_image_embeddings)
clip_index_build_ms = (time.perf_counter() - t0) * 1000

faiss.write_index(clip_index, str(OUTPUT_DIR / "week4_clip_faiss.index"))

print("CLIP vectors indexed:", clip_index.ntotal)
print("Dimension:", clip_dim)
print("Build time (ms):", round(clip_index_build_ms, 3))

In [ ]:
# ============================================================
# 9. ENCODE QUERIES WITH CLIP TEXT ENCODER
# ============================================================

queries = test_queries["query"].fillna("").astype(str).tolist()

clip_query_embeddings_list = []

with torch.no_grad():
    for start in tqdm(range(0, len(queries), CLIP_BATCH_SIZE)):
        batch = queries[start:start+CLIP_BATCH_SIZE]
        inputs = clip_processor(
            text=batch,
            return_tensors="pt",
            padding=True,
            truncation=True
        )
        input_ids = inputs["input_ids"].to(device)
        attention_mask = inputs["attention_mask"].to(device)

        feats = clip_model.get_text_features(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        feats = feats / feats.norm(dim=-1, keepdim=True)
        clip_query_embeddings_list.append(feats.cpu().numpy().astype("float32"))

clip_query_embeddings = np.vstack(clip_query_embeddings_list)
np.save(OUTPUT_DIR / "week4_clip_query_embeddings.npy", clip_query_embeddings)

print("Query embedding shape:", clip_query_embeddings.shape)

In [ ]:
# ============================================================
# 10. EVALUATION HELPERS
# ============================================================

from collections import defaultdict

relevance_lookup = defaultdict(dict)
for row in gold[["query_id","product_id","relevance_label"]].itertuples(index=False):
    relevance_lookup[row.query_id][row.product_id] = int(row.relevance_label)

known_positives = (
    gold[gold["relevance_label"] >= 2]
    .groupby("query_id")["product_id"]
    .apply(lambda x:set(x.astype(str)))
    .to_dict()
)

def recall_at_k(retrieved_ids, positive_ids, k):
    pos = set(positive_ids)
    if not pos:
        return np.nan
    return len(set(retrieved_ids[:k]) & pos) / len(pos)

def judged_metrics(qid, score_by_pid, k=10):
    rows = gold[gold["query_id"] == qid][["product_id","relevance_label"]].copy()
    rows["score"] = rows["product_id"].map(score_by_pid).fillna(-1e9)

    labels = rows["relevance_label"].astype(float).to_numpy()
    scores = rows["score"].astype(float).to_numpy()

    ndcg = float(ndcg_score([labels],[scores],k=min(k,len(rows)))) if len(rows) else np.nan

    ranked = rows.sort_values("score", ascending=False)
    mrr = 0.0
    for rank, rel in enumerate(ranked["relevance_label"].tolist(), start=1):
        if rel > 0:
            mrr = 1.0/rank
            break
    return ndcg, mrr

def summarize(df):
    return {
        "NDCG@10_judged": df["NDCG@10_judged"].mean(),
        "MRR_judged": df["MRR_judged"].mean(),
        "Recall@10_full": df["Recall@10_full"].mean(),
        "Recall@50_full": df["Recall@50_full"].mean(),
        "queries_evaluated": len(df),
    }

In [ ]:
# ============================================================
# 11. CLIP-ONLY IMAGE RETRIEVAL
# ============================================================

clip_top_rows = []
clip_query_metrics = []

t0 = time.perf_counter()
clip_scores_top, clip_rows_top = clip_index.search(clip_query_embeddings, TOP_K)
clip_batch_search_ms = (time.perf_counter() - t0) * 1000

for qi, row in enumerate(test_queries.itertuples(index=False)):
    qid, query = row.query_id, row.query

    clip_rows = clip_rows_top[qi]
    catalog_indices = available_indices[clip_rows]
    retrieved_pids = idx_to_pid[catalog_indices].tolist()

    for rank, (catalog_idx, score) in enumerate(
        zip(catalog_indices, clip_scores_top[qi]), start=1
    ):
        pid = str(idx_to_pid[catalog_idx])
        clip_top_rows.append({
            "model": "CLIP-image",
            "query_id": qid,
            "query": query,
            "rank": rank,
            "product_id": pid,
            "title": title_map.get(pid, ""),
            "relevance_label": relevance_lookup[qid].get(pid, np.nan),
            "score": float(score),
        })

    judged_pids = gold.loc[gold["query_id"] == qid, "product_id"].tolist()

    # Compute CLIP score only where image exists; no-image products get conservative floor.
    qemb = clip_query_embeddings[qi]
    all_clip_for_available = clip_image_embeddings @ qemb
    score_lookup_available = {
        str(idx_to_pid[catalog_idx]): float(score)
        for catalog_idx, score in zip(available_indices, all_clip_for_available)
    }
    floor = float(all_clip_for_available.min()) - 1e-6 if len(all_clip_for_available) else -1e9
    score_by_pid = {pid: score_lookup_available.get(pid, floor) for pid in judged_pids}

    ndcg, mrr = judged_metrics(qid, score_by_pid, 10)
    pos = known_positives.get(qid, set())

    clip_query_metrics.append({
        "model": "CLIP-image",
        "query_id": qid,
        "query": query,
        "NDCG@10_judged": ndcg,
        "MRR_judged": mrr,
        "Recall@10_full": recall_at_k(retrieved_pids, pos, 10),
        "Recall@50_full": recall_at_k(retrieved_pids, pos, 50),
    })

clip_top50 = pd.DataFrame(clip_top_rows)
clip_qmetrics = pd.DataFrame(clip_query_metrics)

display(pd.DataFrame([{"model":"CLIP-image", **summarize(clip_qmetrics)}]))

In [ ]:
# ============================================================
# 12. MULTIMODAL FUSION: SBERT + CLIP
# ============================================================

# Use full 10k SBERT dot products and CLIP image scores for available-image products.
# Both components are mapped to [0,1] per query before fixed-weight fusion.

def minmax(x):
    x = np.asarray(x, dtype=np.float32)
    lo, hi = float(np.min(x)), float(np.max(x))
    if hi - lo < 1e-12:
        return np.zeros_like(x)
    return (x - lo) / (hi - lo)

# V1 SBERT query vectors can be recovered by using the existing Top50 only? No:
# Product embeddings are available, but query embeddings were not saved by V1.
# Recompute SBERT query vectors with the same V1 model.
from sentence_transformers import SentenceTransformer

sbert_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

sbert_query_embeddings = sbert_model.encode(
    queries,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

fusion_top_rows = []
fusion_query_metrics = []

for qi, row in enumerate(tqdm(test_queries.itertuples(index=False), total=len(test_queries))):
    qid, query = row.query_id, row.query

    # Text similarity for all 10k products.
    sbert_raw = sbert_embeddings @ sbert_query_embeddings[qi]
    sbert_norm = minmax(sbert_raw)

    # Image similarity. No-image products receive 0 after normalization.
    clip_raw_available = clip_image_embeddings @ clip_query_embeddings[qi]
    clip_norm_available = minmax(clip_raw_available)

    clip_norm_full = np.zeros(len(products), dtype=np.float32)
    clip_norm_full[available_indices] = clip_norm_available

    fusion = TEXT_WEIGHT * sbert_norm + IMAGE_WEIGHT * clip_norm_full

    top_idx = np.argpartition(-fusion, min(TOP_K, len(fusion))-1)[:TOP_K]
    top_idx = top_idx[np.argsort(-fusion[top_idx])]
    retrieved_pids = idx_to_pid[top_idx].tolist()

    for rank, idx in enumerate(top_idx, start=1):
        pid = str(idx_to_pid[idx])
        fusion_top_rows.append({
            "model": "SBERT+CLIP",
            "query_id": qid,
            "query": query,
            "rank": rank,
            "product_id": pid,
            "title": title_map.get(pid, ""),
            "relevance_label": relevance_lookup[qid].get(pid, np.nan),
            "sbert_score_norm": float(sbert_norm[idx]),
            "clip_score_norm": float(clip_norm_full[idx]),
            "fusion_score": float(fusion[idx]),
            "has_image": bool(products.iloc[idx]["image_available"]),
        })

    judged_pids = gold.loc[gold["query_id"] == qid, "product_id"].tolist()
    score_by_pid = {
        pid: float(fusion[pid_to_idx[pid]])
        for pid in judged_pids if pid in pid_to_idx
    }

    ndcg, mrr = judged_metrics(qid, score_by_pid, 10)
    pos = known_positives.get(qid, set())

    fusion_query_metrics.append({
        "model": "SBERT+CLIP",
        "query_id": qid,
        "query": query,
        "NDCG@10_judged": ndcg,
        "MRR_judged": mrr,
        "Recall@10_full": recall_at_k(retrieved_pids, pos, 10),
        "Recall@50_full": recall_at_k(retrieved_pids, pos, 50),
    })

fusion_top50 = pd.DataFrame(fusion_top_rows)
fusion_qmetrics = pd.DataFrame(fusion_query_metrics)

display(pd.DataFrame([{"model":"SBERT+CLIP", **summarize(fusion_qmetrics)}]))

In [ ]:
# ============================================================
# 13. FINAL COMPARISON
# ============================================================

v1_metrics = pd.read_csv(V1_METRICS_PATH)

clip_summary = {"model":"CLIP-image", **summarize(clip_qmetrics)}
fusion_summary = {"model":"SBERT+CLIP", **summarize(fusion_qmetrics)}

comparison = pd.concat([
    v1_metrics[[
        "model","NDCG@10_judged","MRR_judged",
        "Recall@10_full","Recall@50_full","queries_evaluated"
    ]],
    pd.DataFrame([clip_summary, fusion_summary])
], ignore_index=True)

text_recall50 = comparison.loc[
    comparison["model"] == "SBERT+FAISS", "Recall@50_full"
].iloc[0]

comparison["Recall@50_vs_SBERT_%"] = (
    (comparison["Recall@50_full"] - text_recall50) / text_recall50 * 100
)

print("Final Week 4 comparison:")
display(comparison.round(4))

fusion_recall_gain = comparison.loc[
    comparison["model"] == "SBERT+CLIP", "Recall@50_vs_SBERT_%"
].iloc[0]

print(
    "\nRoadmap multimodal Recall@50 >= 10% above text-only:",
    "PASS" if fusion_recall_gain >= 10 else "NOT YET",
    f"({fusion_recall_gain:+.2f}%)"
)

In [ ]:
# ============================================================
# 14. CLIP / MULTIMODAL LATENCY
# ============================================================

latency_queries = test_queries.head(min(LATENCY_SAMPLE_SIZE, len(test_queries)))
clip_text_ms = []
clip_faiss_ms = []

with torch.no_grad():
    for query in latency_queries["query"].astype(str):
        t0 = time.perf_counter()
        inp = clip_processor(
            text=[query],
            return_tensors="pt",
            padding=True,
            truncation=True
        )
        feats = clip_model.get_text_features(
            input_ids=inp["input_ids"].to(device),
            attention_mask=inp["attention_mask"].to(device)
        )
        feats = feats / feats.norm(dim=-1, keepdim=True)
        q = feats.cpu().numpy().astype("float32")
        clip_text_ms.append((time.perf_counter()-t0)*1000)

        t0 = time.perf_counter()
        _ = clip_index.search(q, TOP_K)
        clip_faiss_ms.append((time.perf_counter()-t0)*1000)

def summarize_latency(name, vals):
    a=np.asarray(vals,dtype=float)
    return {
        "component":name,
        "n_queries":len(a),
        "mean_ms":a.mean(),
        "median_ms":np.median(a),
        "p95_ms":np.percentile(a,95),
        "max_ms":a.max()
    }

clip_latency = pd.DataFrame([
    summarize_latency("CLIP text query embedding", clip_text_ms),
    summarize_latency("CLIP FAISS image Top50 search", clip_faiss_ms),
    summarize_latency(
        "CLIP embedding + image FAISS search",
        np.asarray(clip_text_ms)+np.asarray(clip_faiss_ms)
    ),
])

display(clip_latency.round(3))

In [ ]:
# ============================================================
# 15. ERROR / COVERAGE ANALYSIS
# ============================================================

per_query = clip_qmetrics.merge(
    fusion_qmetrics,
    on=["query_id","query"],
    suffixes=("_CLIP","_FUSION")
)

v1_sbert_q = pd.read_csv(list(EXTRACT_DIR.rglob("week4_sbert_query_metrics.csv"))[0])
v1_sbert_q["query_id"] = v1_sbert_q["query_id"].astype(str)

per_query = per_query.merge(
    v1_sbert_q[[
        "query_id","NDCG@10_judged","Recall@50_full"
    ]],
    on="query_id",
    how="left"
).rename(columns={
    "NDCG@10_judged":"NDCG@10_judged_SBERT",
    "Recall@50_full":"Recall@50_full_SBERT"
})

per_query["NDCG_fusion_minus_SBERT"] = (
    per_query["NDCG@10_judged_FUSION"] - per_query["NDCG@10_judged_SBERT"]
)
per_query["Recall50_fusion_minus_SBERT"] = (
    per_query["Recall@50_full_FUSION"] - per_query["Recall@50_full_SBERT"]
)

print("Largest multimodal NDCG gains:")
display(
    per_query.sort_values("NDCG_fusion_minus_SBERT", ascending=False)
    .head(10)[[
        "query_id","query","NDCG@10_judged_SBERT",
        "NDCG@10_judged_FUSION","NDCG_fusion_minus_SBERT",
        "Recall@50_full_SBERT","Recall@50_full_FUSION"
    ]]
)

print("\nLargest multimodal NDCG losses:")
display(
    per_query.sort_values("NDCG_fusion_minus_SBERT")
    .head(10)[[
        "query_id","query","NDCG@10_judged_SBERT",
        "NDCG@10_judged_FUSION","NDCG_fusion_minus_SBERT",
        "Recall@50_full_SBERT","Recall@50_full_FUSION"
    ]]
)

In [ ]:
# ============================================================
# 16. PLOTS
# ============================================================

metric_cols = [
    "NDCG@10_judged","MRR_judged",
    "Recall@10_full","Recall@50_full"
]

ax = comparison.set_index("model")[metric_cols].T.plot(
    kind="bar", figsize=(11,5)
)
ax.set_title("Week 4 — BM25 vs SBERT vs CLIP vs Multimodal")
ax.set_ylabel("Metric")
ax.set_ylim(0,1.05)
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "week4_v2_multimodal_comparison.png",
    dpi=180, bbox_inches="tight"
)
plt.show()

coverage_df = pd.DataFrame({
    "status":["usable image","missing/failed"],
    "count":[
        int(products["image_available"].sum()),
        int((~products["image_available"]).sum())
    ]
})

ax = coverage_df.set_index("status").plot(
    kind="bar", legend=False, figsize=(7,4)
)
ax.set_title("Week 4 V2 — Product Image Coverage")
ax.set_ylabel("Products")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "week4_v2_image_coverage.png",
    dpi=180, bbox_inches="tight"
)
plt.show()

In [ ]:
# ============================================================
# 17. EXPORT RESULTS
# ============================================================

clip_top50.to_csv(OUTPUT_DIR / "week4_clip_top50.csv", index=False)
fusion_top50.to_csv(OUTPUT_DIR / "week4_multimodal_top50.csv", index=False)

clip_qmetrics.to_csv(OUTPUT_DIR / "week4_clip_query_metrics.csv", index=False)
fusion_qmetrics.to_csv(OUTPUT_DIR / "week4_multimodal_query_metrics.csv", index=False)

comparison.to_csv(OUTPUT_DIR / "week4_v2_final_metrics.csv", index=False)
clip_latency.to_csv(OUTPUT_DIR / "week4_v2_clip_latency.csv", index=False)
per_query.to_csv(OUTPUT_DIR / "week4_v2_per_query_analysis.csv", index=False)

image_coverage = pd.DataFrame([{
    "catalog_products": len(products),
    "nonempty_image_urls": int((products["image_url"].str.len()>0).sum()),
    "successful_images": int(products["image_available"].sum()),
    "successful_coverage_pct": float(products["image_available"].mean()*100),
    "attempt_limit": MAX_IMAGE_DOWNLOADS,
}])
image_coverage.to_csv(OUTPUT_DIR / "week4_v2_image_coverage.csv", index=False)

readme = f'''
Amazon Search / LTR — Week 4 V2
================================

CLIP model: {CLIP_MODEL}
Text weight: {TEXT_WEIGHT:.2f}
Image weight: {IMAGE_WEIGHT:.2f}

Pipeline:
Week 1 product images
-> CLIP image embeddings
-> FAISS image index
-> text-to-image retrieval

Multimodal:
{TEXT_WEIGHT:.2f} * normalized SBERT similarity
+
{IMAGE_WEIGHT:.2f} * normalized CLIP image similarity

Products with no usable image retain text retrieval
and receive no positive image contribution.

Outputs include:
- CLIP embeddings + FAISS index
- CLIP Top-50
- SBERT+CLIP Top-50
- final comparison metrics
- latency
- image coverage
- per-query error analysis
- plots
'''

with open(OUTPUT_DIR / "README_WEEK4_V2.txt","w",encoding="utf-8") as f:
    f.write(readme)

zip_path = shutil.make_archive(
    str(ROOT / "amazon_week4_v2_clip_multimodal_results"),
    "zip",
    OUTPUT_DIR
)

print("Created:", zip_path)

In [ ]:
# ============================================================
# 18. FINAL SCORECARD + DOWNLOAD
# ============================================================

print("="*90)
print("WEEK 4 V2 COMPLETE")
print("="*90)

display(comparison.round(4))
display(image_coverage.round(2))
display(clip_latency.round(3))

fusion_row = comparison[comparison["model"]=="SBERT+CLIP"].iloc[0]
print(
    "Multimodal Recall@50 improvement vs SBERT:",
    f'{fusion_row["Recall@50_vs_SBERT_%"]:+.2f}%'
)

if is_colab:
    files.download(str(ROOT / "amazon_week4_v2_clip_multimodal_results.zip"))
else:
    print(ROOT / "amazon_week4_v2_clip_multimodal_results.zip")